# Postprocessing Pipeline — nnUNet Predictions

**Execution context:** Local — `venv-napari` kernel  
**Pipeline steps:**
1. Concatenate split inference chunks → full-volume `.nii.gz`
2. *(Optional)* Convert to `.mha` via Fiji
3. Validate output (shape, unique labels)
4. Visualize raw volume + prediction in Napari

## 1 — Import Required Libraries

In [ ]:
import sys
import json
import os
from pathlib import Path

import numpy as np
import tifffile as tiff
import SimpleITK as sitk

# Add repo root (for postprocessing_nnUNet_predict*.py, now in 04_inference/)
# and 02_preprocessing/nnunet/ (for __path__.py, moved there in the 2026-07 reorg)
# to sys.path so we can import existing scripts directly.
REPO_ROOT = Path(r"C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT")
INFERENCE_DIR = REPO_ROOT / "04_inference"
NNUNET_PREPROCESSING_DIR = REPO_ROOT / "02_preprocessing" / "nnunet"
for p in (INFERENCE_DIR, NNUNET_PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from postprocessing_nnUNet_predict_concatenate import ensemble_files
from postprocessing_nnUNet_predict import nii_to_mha
from __path__ import PATH_ImageJ

print("Imports OK")
print(f"Repo root: {REPO_ROOT}")
print(f"PATH_ImageJ: '{PATH_ImageJ}' {'(Fiji available)' if PATH_ImageJ else '(Fiji not configured — .mha step will be skipped)'}")

## 2 — Configuration & Path Setup

Edit these paths to point to your sample's data directory.

In [2]:
# ── Sample base directory ─────────────────────────────────────────────────────
SAMPLE_BASE = Path(r"C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem")

# ── Input: split prediction chunks produced by nnUNetv2_predict ───────────────
INF_OUTPUT = SAMPLE_BASE / "inference_output"

# ── Output: reassembled full-volume prediction (created by Step 1) ─────────────
CONCAT_OUTPUT = SAMPLE_BASE / "inference_concatenated"

# ── Output: .mha export (created by Step 2, requires Fiji) ───────────────────
MHA_OUTPUT = SAMPLE_BASE / "inference_mha"

# ── Raw grayscale volume for Napari overlay ───────────────────────────────────
RAW_TIF = SAMPLE_BASE / "images" / "nlm_volume.tif"

# ── Label metadata ────────────────────────────────────────────────────────────
DATASET_INFO = REPO_ROOT / "dataset_info.json"

# Validate key paths exist
for p, label in [(INF_OUTPUT, "inference_output"), (RAW_TIF, "raw TIF"), (DATASET_INFO, "dataset_info.json")]:
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {label}: {p}")

  [OK] inference_output: C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\inference_output
  [OK] raw TIF: C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\images\nlm_volume.tif
  [OK] dataset_info.json: C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT\dataset_info.json


## 3 — Step 1: Concatenate Split Predictions

Calls `postprocessing_nnUNet_predict_concatenate.ensemble_files()` to reassemble the overlapping chunks back into a single full-volume `.nii.gz`.

In [3]:
chunks = sorted(INF_OUTPUT.glob("*.nii.gz"))
print(f"Found {len(chunks)} split chunk(s) in {INF_OUTPUT}:")
for c in chunks:
    print(f"  {c.name}")

if not chunks:
    raise FileNotFoundError(f"No .nii.gz files found in {INF_OUTPUT}")

print(f"\nRunning ensemble_files -> {CONCAT_OUTPUT}")
ensemble_files(str(INF_OUTPUT), str(CONCAT_OUTPUT))

result_files = sorted(CONCAT_OUTPUT.glob("*.nii.gz"))
print(f"\nConcatenation complete. Output files:")
for f in result_files:
    img = sitk.ReadImage(str(f))
    print(f"  {f.name}  |  size (X,Y,Z): {img.GetSize()}  |  spacing: {img.GetSpacing()}")

Found 8 split chunk(s) in C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\inference_output:
  nlm_volume__2__0__114_.nii.gz
  nlm_volume__2__132__278_.nii.gz
  nlm_volume__2__214__360_.nii.gz
  nlm_volume__2__296__442_.nii.gz
  nlm_volume__2__378__524_.nii.gz
  nlm_volume__2__460__606_.nii.gz
  nlm_volume__2__50__196_.nii.gz
  nlm_volume__2__542__652_.nii.gz

Running ensemble_files -> C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\inference_concatenated
nlm_volume: 8 files found

Concatenation complete. Output files:
  nlm_volume.nii.gz  |  size (X,Y,Z): (650, 650, 652)  |  spacing: (1.0, 1.0, 1.0)


## 4 — Step 2 (Optional): Convert to .mha via Fiji

Calls `postprocessing_nnUNet_predict.nii_to_mha()`.  
**Skipped automatically** if `PATH_ImageJ` is empty in `__path__.py`.

In [4]:
if not PATH_ImageJ:
    print("PATH_ImageJ is not set in __path__.py — skipping .mha conversion.")
    print("To enable: set PATH_ImageJ to your Fiji executable in __path__.py")
else:
    print(f"Converting {CONCAT_OUTPUT} -> {MHA_OUTPUT} via Fiji...")
    nii_to_mha(str(CONCAT_OUTPUT), str(MHA_OUTPUT))
    mha_files = sorted(MHA_OUTPUT.glob("*.mha"))
    print(f"Done. {len(mha_files)} .mha file(s) written:")
    for f in mha_files:
        print(f"  {f.name}")

PATH_ImageJ is not set in __path__.py — skipping .mha conversion.
To enable: set PATH_ImageJ to your Fiji executable in __path__.py


## 5 — Load & Validate Postprocessing Results

Load the concatenated prediction and verify shapes, dtype, and label distribution against `dataset_info.json`.

In [8]:
# Load label metadata
with open(DATASET_INFO) as f:
    info = json.load(f)

label_names = {int(k): v for k, v in info["labels"].items()}
colors      = {int(k): np.array(v) / 255.0 for k, v in info["colors"].items()}
# num_classes = real tissue classes (excludes label 0 = ToPredict/ignore)
num_classes = len(label_names) - 1  # 9

# Load raw grayscale
raw = tiff.imread(str(RAW_TIF))
print(f"Raw volume  — shape: {raw.shape}  dtype: {raw.dtype}")

# Load concatenated prediction (SimpleITK → numpy, axis order (Z, Y, X))
pred_path = next(CONCAT_OUTPUT.glob("*.nii.gz"))
pred_sitk  = sitk.ReadImage(str(pred_path))
pred       = sitk.GetArrayFromImage(pred_sitk)   # (Z, Y, X)

# The concatenation script applies img[:, ::-1, :] (SITK Y-flip) to restore the
# original TIF orientation after the Fiji/nibabel conversion flips Y.
# GetArrayFromImage reads the physically-reversed pixel buffer as-is, so we must
# undo that flip here to align the prediction with the raw TIF.
pred = pred[:, ::-1, :]
print(f"Prediction  — shape: {pred.shape}  dtype: {pred.dtype}  (Y-flip undone)")

# Reverse mask_to_nnUNet label shift: nnUNet label i  →  original label i+1
# Clamp so no value exceeds num_classes (9 = otherPores)
pred_remapped = (pred.astype(np.int32) + 1).clip(0, num_classes).astype(np.uint8)

assert raw.shape == pred_remapped.shape, (
    f"Shape mismatch: raw {raw.shape} vs prediction {pred_remapped.shape}"
)
print("\nShape check: OK")

# Label distribution
print("\nLabel distribution (prediction):")
print(f"  {'ID':>4}  {'Name':<20}  {'Voxels':>12}  {'%':>6}")
print(f"  {'-'*4}  {'-'*20}  {'-'*12}  {'-'*6}")
total = pred_remapped.size
for uid in sorted(np.unique(pred_remapped)):
    count = np.sum(pred_remapped == uid)
    name  = label_names.get(uid, "unknown")
    print(f"  {uid:>4}  {name:<20}  {count:>12,}  {100*count/total:>5.1f}%")

Raw volume  — shape: (652, 650, 650)  dtype: float32
Prediction  — shape: (652, 650, 650)  dtype: uint8  (Y-flip undone)

Shape check: OK

Label distribution (prediction):
    ID  Name                        Voxels       %
  ----  --------------------  ------------  ------
     1  Matrix                 232,260,309   84.3%
     3  Rocks                    7,507,640    2.7%
     4  FreshRoots                 477,228    0.2%
     6  otherPOM                35,224,823   12.8%


## 5b — Save Prediction as TIFF

In [11]:
pred_tif_path = CONCAT_OUTPUT / (pred_path.stem.replace(".nii", "") + "_labels.tif")
tiff.imwrite(str(pred_tif_path), pred_remapped)
print(f"Prediction saved as TIFF: {pred_tif_path}")
print(f"  shape: {pred_remapped.shape}  dtype: {pred_remapped.dtype}")

Prediction saved as TIFF: C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\inference_concatenated\nlm_volume_labels.tif
  shape: (652, 650, 650)  dtype: uint8


## 6 — Visualize Predictions with Napari

Opens an interactive Napari viewer with:
- **Layer 1** – raw grayscale volume (gray colormap)
- **Layer 2** – prediction label overlay (colors from `dataset_info.json`, additive blend)

Label mapping applied before display: nnUNet output label $i$ → `dataset_info.json` label $i+1$  
(reverses the `mask_to_nnUNet` −1 shift applied during training prep)

In [6]:
%gui qt

In [9]:
import napari

viewer = napari.Viewer()

viewer.add_image(
    raw,
    name="nlm_volume (raw)",
    colormap="gray",
    blending="additive",
)

labels_layer = viewer.add_labels(
    pred_remapped,
    name="Prediction",
    opacity=0.5,
    blending="additive",
)

# Apply dataset_info.json colors (normalized to [0, 1])
labels_layer.color = colors

print("Napari viewer opened.")
print(f"  Raw layer  : {raw.shape}  {raw.dtype}")
print(f"  Label layer: {pred_remapped.shape}  {pred_remapped.dtype}")
print(f"  Labels shown: { {uid: label_names[uid] for uid in np.unique(pred_remapped)} }")

Napari viewer opened.
  Raw layer  : (652, 650, 650)  float32
  Label layer: (652, 650, 650)  uint8
  Labels shown: {np.uint8(1): 'Matrix', np.uint8(3): 'Rocks', np.uint8(4): 'FreshRoots', np.uint8(6): 'otherPOM'}
